# Fusion Oncology: AI-Powered Cancer Target Discovery

### Author: [Kevin Thomas](mailto:ket189@pitt.edu)
### GitHub: [https://github.com/mytechnotalent/fusion_oncology](https://github.com/mytechnotalent/fusion_oncology)

## Overview

This notebook demonstrates **Fusion Oncology** on the **GDSC dataset** - combining:

- **XGBoost** feature importance (which genes discriminate cancer types)
- **DNABERT-2** sequence embeddings (structural gene fragility)
- **Multi-omics** integration (mutations, CNAs, methylation)
- **Clinical evidence** aggregation (OpenTargets, CIViC, ClinicalTrials.gov)
- **Drug-target** mapping and resistance prediction
- **Digital twin** tumor simulations

## Dataset

**GDSC: Genomics of Drug Sensitivity in Cancer**
- **Source**: 1,002 cancer cell lines with genomic features
- **Add to Kaggle**: [GDSC Dataset](https://www.kaggle.com/datasets/samiraalipour/genomics-of-drug-sensitivity-in-cancer-gdsc)
- **Input**: `/kaggle/input/genomics-of-drug-sensitivity-in-cancer-gdsc/`
- **Output**: `/kaggle/working/`
- **Note**: Cell lines are lab-adapted models for workflow demonstration

## Step 1: Setup and Installation

In [ ]:
# ── Environment Variables (must be set BEFORE any library import) ─────────────
# Kaggle pre-installs TensorFlow, JAX, and XLA which emit dozens of noisy C++
# registration warnings to stderr on GPU runtimes.  The env-vars below silence
# them at the earliest possible point — before Python even imports the libs.
import os

# 1. TensorFlow C++ log level: 0=INFO, 1=WARNING, 2=ERROR, 3=FATAL (suppress all)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# 2. XLA verbose log level: suppresses "computation placer already registered"
#    messages that TF_CPP_MIN_LOG_LEVEL alone does not cover
os.environ["TF_CPP_MIN_VLOG_LEVEL"] = "3"

# 3. Tell HuggingFace Transformers to skip TensorFlow entirely so it never
#    triggers the TF/XLA init path (we only need the PyTorch backend)
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

# 4. Disable the pydevd debugger file-validation check that prints an
#    informational note on Kaggle kernels (not an error, just noise)
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

# ── Install Fusion Oncology ──────────────────────────────────────────────────
# Installs the package directly from the GitHub main branch along with
# openpyxl (needed to read the GDSC .xlsx cell-line metadata file).
%pip install -q "openpyxl>=3.1,<4" "fusion-oncology @ git+https://github.com/mytechnotalent/fusion_oncology.git@main"
print("Installation complete!")

In [ ]:
# Suppress ALL noisy third-party warnings
import warnings
import logging
import logging as _logging
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, message=".*Unable to import Triton.*")
warnings.filterwarnings("ignore", message=".*not initialized.*")
warnings.filterwarnings("ignore", message=".*Some weights.*were not initialized.*")
warnings.filterwarnings("ignore", message=".*Precision is ill-defined.*")
warnings.filterwarnings("ignore", message=".*UndefinedMetricWarning.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*Unknown extension.*")

# Silence sklearn UndefinedMetricWarning globally
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

# Silence absl, transformers, and fusion_oncology loggers
logging.getLogger("absl").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
logging.getLogger("transformers.integrations.tensor_parallel").setLevel(logging.CRITICAL)
logging.getLogger("fusion_oncology").setLevel(logging.WARNING)

# Import all libraries
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Fusion Oncology imports
from fusion_oncology.config import ProjectConfig
from fusion_oncology.models.fusion import FusionEngine
from fusion_oncology.analysis.clinical_evidence import ClinicalEvidenceAggregator
from fusion_oncology.analysis.drug_target import DrugTargetMapper
from fusion_oncology.analysis.resistance import ResistancePredictor
from fusion_oncology.models.digital_twin import (
    DigitalTwin,
    SimulationConfig,
    DrugRegimen,
)
from fusion_oncology.models.companion_dx import PatientProfile, CompanionDiagnostic
from fusion_oncology.analysis.pathway import PathwayEnrichment

## Step 2: Load GDSC Dataset

Loading cancer cell line data with genomic features and tissue classifications.

In [ ]:
# 1. Configure project settings (production-grade defaults)
# - top_k_genes: 15 top therapeutic targets for richer fusion signal
# - fuzz_iterations: 10 DNABERT-2 mutation rounds per gene
# - xgb_n_estimators: 1000 boosting rounds with lower LR for convergence
# - xgb_max_depth: 6 shallower trees to reduce over-fitting
# - xgb_learning_rate: 0.03 slower learning for better generalisation
# - xgb_min_child_weight: 5 prevents over-fitting sparse splits
# - xgb_gamma: 0.2 minimum loss reduction for further partitioning
# - xgb_reg_alpha: 0.3 L1 regularisation on leaf weights
# - xgb_reg_lambda: 2.0 L2 regularisation on leaf weights
# - min_class_size: 40 merge rare cancer types into OTHER
# - enable_hpo: False (set to True to run 50-trial Optuna Bayesian search)
# - output_dir: Kaggle working directory for results
#
# NEW in this version:
# - Intelligent class merging: cancer types with < 40 cell lines → "OTHER"
# - Feature engineering: 10 row-level distributional features per sample
# - Repeated stratified 5-fold CV (3 repeats) for stable metric estimates
# - Optional Optuna HPO: set enable_hpo=True for Bayesian hyperparameter search
cfg = ProjectConfig(
    top_k_genes=15,
    fuzz_iterations=10,
    xgb_n_estimators=1000,
    xgb_max_depth=6,
    xgb_learning_rate=0.03,
    xgb_min_child_weight=5,
    xgb_gamma=0.2,
    xgb_reg_alpha=0.3,
    xgb_reg_lambda=2.0,
    min_class_size=40,
    enable_hpo=False,
    output_dir=Path("/kaggle/working/results"),
)
print(f"Output directory: /kaggle/working/results")
print(f"Top-K genes: {cfg.top_k_genes}")
print(f"XGBoost: {cfg.xgb_n_estimators} trees, depth {cfg.xgb_max_depth}, lr {cfg.xgb_learning_rate}")
print(f"Class merging: types < {cfg.min_class_size} samples → OTHER")
print(f"Optuna HPO: {'ENABLED' if cfg.enable_hpo else 'disabled (set enable_hpo=True to activate)'}")

In [ ]:
# GDSC Input paths (exact file names from the Kaggle dataset):
GDSC_DIR = "/kaggle/input/genomics-of-drug-sensitivity-in-cancer-gdsc"
cell_lines = f"{GDSC_DIR}/Cell_Lines_Details.xlsx"
gdsc_main = f"{GDSC_DIR}/GDSC_DATASET.csv"
gdsc2 = f"{GDSC_DIR}/GDSC2-dataset.csv"
compounds = f"{GDSC_DIR}/Compounds-annotation.csv"
print("GDSC dataset files:")
print(f"  1. GDSC_DATASET.csv      (main merged dataset)")
print(f"  2. GDSC2-dataset.csv     (raw drug sensitivity, IC50)")
print(f"  3. Cell_Lines_Details.xlsx (cell line metadata)")
print(f"  4. Compounds-annotation.csv (drug targets/pathways)")

In [ ]:
# 2. Load all 4 GDSC files
df_main = pd.read_csv(gdsc_main)
df_gdsc2 = pd.read_csv(gdsc2)
df_compounds = pd.read_csv(compounds)
df_cellinfo = pd.read_excel(cell_lines, sheet_name="Cell line details")
print("=== GDSC_DATASET.csv (main merged file) ===")
print(f"Shape: {df_main.shape[0]} rows x {df_main.shape[1]} columns")
print(f"Columns: {', '.join(df_main.columns.tolist())}\n")
print("=== GDSC2-dataset.csv (raw drug sensitivity) ===")
print(f"Shape: {df_gdsc2.shape[0]} rows x {df_gdsc2.shape[1]} columns")
print(f"Columns: {', '.join(df_gdsc2.columns.tolist())}\n")
print("=== Compounds-annotation.csv (drug info) ===")
print(f"Shape: {df_compounds.shape[0]} rows x {df_compounds.shape[1]} columns")
print(f"Columns: {', '.join(df_compounds.columns.tolist())}\n")
print("=== Cell_Lines_Details.xlsx (cell line metadata) ===")
print(f"Shape: {df_cellinfo.shape[0]} rows x {df_cellinfo.shape[1]} columns")
print(f"Columns: {', '.join(df_cellinfo.columns.tolist())}\n")

# 3. Build gene-level feature matrix by pivoting drug sensitivity data.
# Each row = one cell line, each column = a drug target gene,
# values = mean LN_IC50 sensitivity (lower = more sensitive).
# The TARGET column contains gene symbols (EGFR, BRAF, etc.)
# that FusionEngine can use for DNABERT-2 sequence analysis.
print("Building gene-level feature matrix from drug target sensitivity...")

# 4. Split multi-target entries (e.g. "EGFR, ERBB2") into separate rows
df_targets = df_main[["COSMIC_ID", "TCGA_DESC", "TARGET", "LN_IC50"]].dropna(
    subset=["TARGET", "LN_IC50", "TCGA_DESC"]
)
df_exploded = df_targets.assign(TARGET=df_targets["TARGET"].str.split(", ")).explode("TARGET")
df_exploded["TARGET"] = df_exploded["TARGET"].str.strip()

# 5. Pivot: rows=cell lines, columns=target genes, values=mean LN_IC50
pivot = df_exploded.pivot_table(
    index="COSMIC_ID", columns="TARGET", values="LN_IC50", aggfunc="mean"
)

# 6. Get TCGA_DESC for each cell line (majority label)
cell_labels = (
    df_main.dropna(subset=["TCGA_DESC"])
    .groupby("COSMIC_ID")["TCGA_DESC"]
    .agg(lambda x: x.mode().iloc[0])
)

# 7. Align and clean
common_ids = pivot.index.intersection(cell_labels.index)
X = pivot.loc[common_ids].fillna(0)
y = cell_labels.loc[common_ids]

# 8. Filter to genes appearing in at least 10 cell lines
gene_coverage = (X != 0).sum()
valid_genes = gene_coverage[gene_coverage >= 10].index
X = X[valid_genes]

# 9. Keep only real gene/protein symbols (uppercase identifiers like EGFR, BRAF,
# AKT1).  Removes descriptive drug-class terms ("Retinoic acid", "Pyrimidine
# antimetabolite", "Microtubule destabiliser", "others", "Metabolism", etc.)
# that are NOT valid gene symbols and would break downstream pathway/drug lookups.
_GENE_RE = re.compile(r"^[A-Z][A-Z0-9/.:-]*(?:\s*\([^)]+\))?$")
gene_cols = [c for c in X.columns if _GENE_RE.match(c)]
n_dropped = len(X.columns) - len(gene_cols)
X = X[gene_cols]
print(f"Kept {len(gene_cols)} gene symbols, dropped {n_dropped} non-gene targets")

# 10. Merge rare cancer types into "OTHER" to reduce class count.
# Cancer types with fewer than 40 cell lines are too small for robust
# stratified CV and drag down weighted precision/recall/F1.  Merging them
# into a single "OTHER" bucket reduces the problem from ~20+ classes to
# ~5-7 well-represented ones, which is the single biggest metric lift.
MIN_CLASS_SIZE = 40
type_counts = y.value_counts()
rare_types = type_counts[type_counts < MIN_CLASS_SIZE].index
n_rare = len(rare_types)
y = y.replace({t: "OTHER" for t in rare_types})
print(f"Merged {n_rare} rare cancer types (< {MIN_CLASS_SIZE} cell lines) into OTHER")

# 11. Remove the OTHER bucket if it is still too small after merging
other_count = (y == "OTHER").sum()
if other_count < MIN_CLASS_SIZE:
    mask = y != "OTHER"
    X = X.loc[mask]
    y = y.loc[mask]
    print(f"Dropped OTHER bucket ({other_count} cell lines < {MIN_CLASS_SIZE})")

# 12. Print final dataset summary
print(f"\nFeature matrix: {X.shape[0]} cell lines x {X.shape[1]} features")
print(f"Labels: TCGA_DESC ({y.nunique()} cancer types)")
print(f"Class distribution:\n{y.value_counts().to_string()}")
print(f"Sample gene columns: {', '.join(X.columns[:10])}")

## Step 3: Exploratory Data Analysis

In [ ]:
# Cancer type distribution
fig, ax = plt.subplots(figsize=(10, 6))
y.value_counts().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Cancer Type Distribution", fontsize=14, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# Top variable features
top_features = X.var().nlargest(50).index
sample_subset = X.sample(n=100, random_state=42) if len(X) > 100 else X
print(f"Most variable features: {', '.join(str(c) for c in top_features[:10])}")

In [ ]:
# Feature variance heatmap
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    sample_subset[top_features].T,
    cmap="RdBu_r",
    center=0,
    cbar_kws={"label": "Value"},
    ax=ax,
)
ax.set_title("Top 50 Variable Features (GDSC)", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# GDSC drug targets and pathways from Compounds-annotation.csv
print("=== Drug Targets (from Compounds-annotation.csv) ===")
if "TARGET_PATHWAY" in df_compounds.columns:
    print(f"Unique drugs: {df_compounds['DRUG_NAME'].nunique()}")
    print(f"Unique targets: {df_compounds['TARGET'].nunique()}")
    print(f"\nTop target pathways:")
    print(df_compounds["TARGET_PATHWAY"].value_counts().head(10).to_string())
elif "PATHWAY_NAME" in df_compounds.columns:
    print(df_compounds["PATHWAY_NAME"].value_counts().head(10).to_string())

In [ ]:
# Drug sensitivity distribution from GDSC2
if "LN_IC50" in df_gdsc2.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(df_gdsc2["LN_IC50"].dropna(), bins=50, color="steelblue")
    ax.set_title("LN_IC50 Distribution", fontsize=14, fontweight="bold")
    ax.set_xlabel("LN_IC50 (lower = more sensitive)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig, ax = plt.subplots(figsize=(12, 6))
    top_drugs = df_gdsc2["DRUG_NAME"].value_counts().head(15)
    ax.barh(top_drugs.index, top_drugs.values, color="coral")
    ax.set_title("Most Tested Drugs", fontsize=14, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

## Step 4: Run Fusion Analysis

This runs the **production-grade multi-modal fusion model** with maximum optimisation:
1. **Class merging** — cancer types with < 40 cell lines are merged into "OTHER" for cleaner discrimination
2. **Feature engineering** — 10 row-level distributional features (mean, std, skew, kurtosis, IQR, CV, etc.) appended per sample
3. **XGBoost baseline** — 1,000-tree classifier (depth 6, lr 0.03) on drug sensitivity + engineered features, extract top-15 gene importances
4. **DNABERT-2 embeddings** — compute 768-dim sequence embeddings for top-15 genes
5. **Fusion features** — weight each gene's DNABERT-2 embedding by cell-line drug sensitivity → 768-dim genomic context per sample
6. **Fusion XGBoost** — single model on concatenated (drug sensitivity + engineered features + 768 DNABERT-2 dims) feature space
7. **Repeated stratified CV** — 5-fold × 3 repeats for stable Accuracy, Precision, Recall, F1, F2, ROC AUC estimates
8. **(Optional) Optuna HPO** — 50-trial Bayesian hyperparameter search if `enable_hpo=True`

In [ ]:
# Run fusion analysis
print("Running Fusion Analysis...\n")
engine = FusionEngine(cfg)
results = engine.run(X, y)
print("Analysis complete!")

In [ ]:
# Display top targets
print("\n" + "=" * 60)
print("TOP THERAPEUTIC TARGETS")
print("=" * 60)
print(results[["Gene", "Fusion_Index"]].to_string(index=False))
print("=" * 60)

In [ ]:
# Display Fusion Model cross-validation metrics (XGBoost + DNABERT-2 combined)
print("\n" + "=" * 60)
print("FUSION MODEL CROSS-VALIDATION METRICS (5-fold stratified)")
print("XGBoost trained on drug sensitivity + DNABERT-2 embeddings")
print("=" * 60)
fcv = engine.fusion_cv_metrics
for label, key in [
    ("Accuracy", "accuracy"),
    ("Precision", "precision"),
    ("Recall", "recall"),
    ("F1-Score", "f1"),
    ("F2-Score", "f2"),
    ("ROC AUC", "roc_auc"),
]:
    mean = fcv.get(f"mean_{key}", 0)
    std = fcv.get(f"std_{key}", 0)
    print(f"  {label:>10s}: {mean:.4f} ± {std:.4f}")
print("=" * 60)

In [ ]:
# Display supporting DNABERT-2 embedding statistics
print("\n" + "=" * 60)
print("DNABERT-2 EMBEDDING STATISTICS (supporting info)")
print("=" * 60)
dm = engine.dnabert_metrics
print(f"  Model: DNABERT-2-117M (768-dim embeddings)")
print(f"  Genes Analyzed: {dm.get('n_genes_scored', 0)}")
print(f"  Mean Mutation Sensitivity: {dm.get('mean_instability', 0):.6f}")
print(f"  Std  Mutation Sensitivity: {dm.get('std_instability', 0):.6f}")
print(f"  Max  Mutation Sensitivity: {dm.get('max_instability', 0):.6f}")
print(f"  Min  Mutation Sensitivity: {dm.get('min_instability', 0):.6f}")
print(f"  Instability Range: {dm.get('instability_range', 0):.6f}")
print(f"  Signal-to-Noise Ratio: {dm.get('signal_noise_ratio', 0):.2f}")
print("=" * 60)

In [ ]:
# Display Fusion Pipeline evaluation metrics
print("\n" + "=" * 60)
print("FUSION PIPELINE EVALUATION")
print("=" * 60)
dm = engine.dnabert_metrics
n_hits = dm.get("n_driver_hits", 0)
n_total = dm.get("n_genes_scored", 0)
pct = dm.get("driver_enrichment", 0) * 100
drivers_found = dm.get("driver_genes_found", [])
print(f"  Cancer Driver Enrichment: {n_hits}/{n_total} ({pct:.0f}%)")
if drivers_found:
    print(f"  Known Drivers Found: {', '.join(drivers_found)}")
else:
    print(f"  Known Drivers Found: (novel/research targets)")
print(f"  Reference: COSMIC Cancer Gene Census ({len(engine._CANCER_DRIVERS)} genes)")
fi_values = results["Fusion_Index"]
print(f"  Top Fusion Index: {fi_values.max():.4f}")
print(f"  Fusion Index Spread: {fi_values.max() - fi_values.min():.4f}")
print("=" * 60)

## Step 5: Clinical Evidence and Drug Mapping

In [ ]:
# Get top gene
top_gene = results.iloc[0]["Gene"]
print(f"Clinical Evidence for {top_gene}:")

In [ ]:
# Query evidence sources
agg = ClinicalEvidenceAggregator(cfg)
evidence = agg.profile(top_gene)

In [ ]:
# Display evidence scores
print(f"  OpenTargets: {evidence.get('opentargets', {}).get('overall_score', 0):.3f}")
print(f"  Clinical Trials: {len(evidence.get('trials', []))}")
print(f"  CIViC Evidence: {len(evidence.get('civic', []))}")
print(f"  Composite Score: {evidence.get('evidence_score', 0):.3f}")

In [ ]:
# Map to available drugs
mapper = DrugTargetMapper(cfg)
drug_df = mapper.annotate(results)

In [ ]:
# Display drug targets
print(f"\nDrug Targets for {top_gene}:")
drug_matches = drug_df[drug_df["Gene"] == top_gene]["Approved_Drugs"].iloc[0]
if drug_matches and drug_matches != "\u2014":
    for drug in drug_matches.split("; "):
        print(f"  - {drug}")

## Step 6: Resistance Mechanisms

In [ ]:
# Query resistance mechanisms
predictor = ResistancePredictor(cfg)
resistance_report = predictor.full_report([top_gene])

In [ ]:
# Display resistance info
print(f"Resistance Mechanisms for {top_gene}:")
if not resistance_report.empty:
    for _, row in resistance_report.iterrows():
        print(f"  {row['Mechanism']} - {row['Risk_Score']}")
else:
    print("  No known resistance")

## Step 7: Digital Twin Tumor Simulation

In [ ]:
# Initialize digital twin
# immune_kill_rate=1e-9 produces a realistic gradual treatment response.
# The default (0.001) with 1e6 immune cells creates an unrealistically high
# kill rate (~1000 per sensitive cell per day), wiping out cells in 1 step.
sim_cfg = SimulationConfig(simulation_days=180, immune_kill_rate=1e-9)
twin = DigitalTwin(sim_config=sim_cfg, project_config=cfg)

In [ ]:
# Add treatment regimen
twin.add_regimen(
    DrugRegimen(name="Targeted Therapy", efficacy=0.15, resistance_rate=0.001, duration_days=180)
)

In [ ]:
# Run simulation
trajectory = twin.simulate()
summary = twin.summary()
print(f"RECIST Response: {summary['recist']}")

In [ ]:
# Plot tumor volume
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trajectory["day"], trajectory["total"], color="darkred", linewidth=2)
ax.set_title("Tumor Volume Over Time", fontweight="bold")
ax.set_xlabel("Days")
ax.set_ylabel("Total Cell Count")
ax.set_yscale("log")
ax.grid(axis="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot cell populations
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    trajectory["day"],
    trajectory["sensitive"],
    label="Sensitive",
    color="green",
    linewidth=2,
)
ax.plot(
    trajectory["day"],
    trajectory["resistant"],
    label="Resistant",
    color="red",
    linewidth=2,
)
ax.set_title("Cell Populations Over Time", fontweight="bold")
ax.set_xlabel("Days")
ax.set_ylabel("Cell Count")
ax.set_yscale("log")
ax.legend()
ax.grid(axis="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Display simulation results
print(f"Best Response: {summary['best_response']['response_pct']:.1f}% reduction")
print(f"Final Volume: {summary['final_tumour']:.2e} mm3")

## Step 8: Companion Diagnostics (Patient-Specific)

In [ ]:
# Create example patient profile (using GDSC tissue type)
patient = PatientProfile(
    patient_id="GDSC-001",
    cancer_type=str(y.iloc[0]),  # Use actual GDSC tissue classification
    mutations=[{"gene": "EGFR", "variant": "L858R", "vaf": 0.42}],
)

In [ ]:
# Generate treatment report
dx = CompanionDiagnostic(cfg)
report = dx.analyse(patient)

In [ ]:
# Display top recommendations
print("TOP RECOMMENDATIONS:")
for i, rec in enumerate(report["treatment_plan"][:3], 1):
    print(f"{i}. {rec['therapy']} - {rec['confidence']:.1%} confidence")
    print(f"   Type: {rec['type']}")

## Step 9: Visualization Dashboard

In [ ]:
# Fusion Index ranking
results_sorted = results.sort_values("Fusion_Index", ascending=True)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(results_sorted["Gene"], results_sorted["Fusion_Index"], color="steelblue")

# Add value labels on each bar
for bar, val in zip(bars, results_sorted["Fusion_Index"]):
    ax.text(
        bar.get_width() + ax.get_xlim()[1] * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}",
        va="center",
        fontsize=10,
        fontweight="bold",
    )

# Customize plot
ax.set_xlabel("Fusion Index (Importance \u00d7 Instability \u00d7 1000)", fontsize=11)
ax.set_title("Top Therapeutic Targets by Fusion Index", fontweight="bold", fontsize=13)
ax.margins(x=0.25)  # Extra room for labels
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Get pathway data
pathway_enricher = PathwayEnrichment(cfg)
pathway_df = pathway_enricher.annotate(results)

In [ ]:
# Count pathway occurrences
all_pathways = []
for pw_list in pathway_df["Pathways"]:
    if pw_list != "\u2014":
        all_pathways.extend(pw_list.split("; "))
pathway_counts = pd.Series(all_pathways).value_counts().head(10)

In [ ]:
# Plot pathway enrichment
fig, ax = plt.subplots(figsize=(10, 5))
if not pathway_counts.empty:
    ax.barh(pathway_counts.index, pathway_counts.values, color="mediumseagreen")
    ax.set_xlabel("Gene Count")
else:
    ax.text(
        0.5,
        0.5,
        "No pathway matches for top genes",
        transform=ax.transAxes,
        ha="center",
        va="center",
        fontsize=14,
        color="gray",
    )
ax.set_title("Top Cancer Pathways", fontweight="bold")
plt.tight_layout()
plt.show()
print("Dashboard complete!")

In [ ]:
# Drug availability
drug_counts = drug_df["Approved_Drugs"].apply(lambda x: 0 if x == "\u2014" else len(x.split("; ")))
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(drug_df["Gene"], drug_counts, color="coral")
ax.set_title("Available Drugs per Target", fontweight="bold")
ax.set_xlabel("Gene Target")
ax.set_ylabel("Number of Approved Drugs")
if drug_counts.sum() == 0:
    ax.text(
        0.5,
        0.5,
        "No approved drugs matched for top targets",
        transform=ax.transAxes,
        ha="center",
        va="center",
        fontsize=14,
        color="gray",
    )
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Importance vs Instability
fig, ax = plt.subplots(figsize=(10, 6))

# Scale bubble size so they are visible even with small Fusion_Index values
max_fi = results["Fusion_Index"].max()
sizes = (results["Fusion_Index"] / max(max_fi, 1e-6)) * 500 + 50
scatter = ax.scatter(
    results["XGB_Importance"],
    results["Instability"],
    s=sizes,
    c=results["Fusion_Index"],
    cmap="viridis",
    alpha=0.7,
    edgecolors="black",
    linewidth=0.5,
)

# Annotate each gene on the scatter
for _, row in results.iterrows():
    ax.annotate(
        row["Gene"],
        (row["XGB_Importance"], row["Instability"]),
        fontsize=9,
        fontweight="bold",
        ha="left",
        va="bottom",
        xytext=(5, 5),
        textcoords="offset points",
    )
ax.set_xlabel("XGBoost Feature Importance", fontsize=12)
ax.set_ylabel("DNABERT-2 Instability Score", fontsize=12)
ax.set_title("Importance vs Instability (bubble = Fusion Index)", fontweight="bold")
plt.colorbar(scatter, label="Fusion Index")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 10: Complete Inference Pipeline (End-to-End Demo)

**This cell runs the ENTIRE workflow on GDSC data:**
1. Load cell line sample
2. Run fusion analysis (XGBoost + DNABERT-2 combined model)
3. Fusion model evaluation metrics (Accuracy, Precision, Recall, F1, F2, ROC AUC)
4. Clinical evidence aggregation
5. Drug-target mapping
6. Resistance prediction
7. Digital twin simulation
8. Companion diagnostics report
9. Cancer driver enrichment

In [ ]:
# Suppress ALL library-level logging during pipeline execution
_prev_root_level = _logging.root.level
_logging.root.setLevel(_logging.CRITICAL)
for _h in _logging.root.handlers:
    _h.setLevel(_logging.CRITICAL)
_logging.getLogger("fusion_oncology").setLevel(_logging.CRITICAL)
_logging.getLogger("transformers").setLevel(_logging.CRITICAL)

# FINAL INTRO
print("=" * 70)
print("FUSION ONCOLOGY - COMPLETE INFERENCE PIPELINE")
print("=" * 70)

# STEP 1: Load and analyze cell line sample from GDSC
print("\n[1/9] Loading GDSC cell line sample...")
sample_cell_line = X.iloc[0:1]  # Get first cell line as example
true_tissue = y.iloc[0]
print(f"Cell line sample loaded | Tissue type: {true_tissue}")

# STEP 2: Run Fusion Analysis
print("\n[2/9] Running Fusion Analysis (XGBoost + DNABERT-2)...")
engine_demo = FusionEngine(cfg)
fusion_results = engine_demo.run(X, y)
top_genes = fusion_results["Gene"].tolist()
print(f"Top {len(top_genes)} therapeutic targets: {', '.join(top_genes)}")
for _, row in fusion_results.iterrows():
    print(
        f"  {row['Gene']:>10s}  FI={row['Fusion_Index']:.4f}  "
        f"(imp={row['XGB_Importance']:.4f}, inst={row['Instability']:.4f})"
    )

# STEP 3: Fusion Model Evaluation Metrics (XGBoost + DNABERT-2 combined)
print("\n[3/9] Fusion model cross-validation metrics (XGBoost + DNABERT-2):")
fcv = engine_demo.fusion_cv_metrics
for label, key in [
    ("Accuracy", "accuracy"),
    ("Precision", "precision"),
    ("Recall", "recall"),
    ("F1-Score", "f1"),
    ("F2-Score", "f2"),
    ("ROC AUC", "roc_auc"),
]:
    mean = fcv.get(f"mean_{key}", 0)
    std = fcv.get(f"std_{key}", 0)
    print(f"  {label:>10s}: {mean:.4f} ± {std:.4f}")

# STEP 4: Clinical Evidence Aggregation
print("\n[4/9] Querying clinical evidence databases...")
agg_demo = ClinicalEvidenceAggregator(cfg)
for gene in top_genes[:3]:
    ev = agg_demo.profile(gene)
    ot_score = ev.get("opentargets", {}).get("overall_score", 0)
    n_trials = len(ev.get("trials", []))
    n_civic = len(ev.get("civic", []))
    composite = ev.get("evidence_score", 0)
    print(
        f"  {gene:>10s}: OT={ot_score:.3f}  Trials={n_trials}  "
        f"CIViC={n_civic}  Composite={composite:.3f}"
    )
# Store top gene evidence for summary
gene_evidence = agg_demo.profile(top_genes[0])

# STEP 5: Drug-Target Mapping
print("\n[5/9] Mapping to FDA-approved drugs...")
mapper_demo = DrugTargetMapper(cfg)
drug_results = mapper_demo.annotate(fusion_results)
any_drug_found = False
for _, row in drug_results.iterrows():
    drugs = row["Approved_Drugs"]
    if drugs and drugs != "—":
        print(f"  {row['Gene']:>10s}: {drugs}")
        any_drug_found = True
if not any_drug_found:
    print("  No approved drugs matched for top targets (novel/research targets)")
top_drugs = drug_results.iloc[0]["Approved_Drugs"]

# STEP 6: Resistance Prediction
print("\n[6/9] Analyzing resistance mechanisms...")
resist_demo = ResistancePredictor(cfg)
resist_report = resist_demo.full_report(top_genes)
known = resist_report[resist_report["Mechanism"] != "None catalogued"]
if not known.empty:
    for _, row in known.iterrows():
        print(f"  {row['Gene']:>10s}: {row['Mechanism']} ({row['Frequency']})")
else:
    print("  No catalogued resistance mechanisms for top targets")
print(f"  Total entries: {len(resist_report)}")

# STEP 7: Digital Twin Simulation
print("\n[7/9] Simulating treatment response (180 days)...")
twin_demo = DigitalTwin(
    sim_config=SimulationConfig(simulation_days=180, immune_kill_rate=1e-9),
    project_config=cfg,
)
twin_demo.add_regimen(
    DrugRegimen(name="Targeted Therapy", efficacy=0.15, resistance_rate=0.001, duration_days=180)
)
traj = twin_demo.simulate()
sim_summary = twin_demo.summary()
print(f"RECIST Response: {sim_summary['recist']}")
print(f"  Best response: {sim_summary['best_response']['response_pct']:.1f}% reduction")

# STEP 8: Companion Diagnostics Report
print("\n[8/9] Generating companion diagnostics...")
test_profile = PatientProfile(
    patient_id="GDSC-DEMO-001",
    cancer_type=str(true_tissue),
    mutations=[{"gene": top_genes[0], "variant": "V600E", "vaf": 0.35}],
)
dx_demo = CompanionDiagnostic(cfg)
dx_report = dx_demo.analyse(test_profile)
n_recs = len(dx_report.get("treatment_plan", []))
print(f"Treatment recommendations: {n_recs}")
if n_recs:
    for i, rec in enumerate(dx_report["treatment_plan"][:3], 1):
        print(f"  {i}. {rec['therapy']} ({rec['confidence']:.0%} confidence)")

# STEP 9: Cancer Driver Enrichment
print("\n[9/9] Cancer driver enrichment analysis:")
dm = engine_demo.dnabert_metrics
n_hits = dm.get("n_driver_hits", 0)
n_total = dm.get("n_genes_scored", 0)
pct = dm.get("driver_enrichment", 0) * 100
drivers_found = dm.get("driver_genes_found", [])
print(f"  Cancer Driver Enrichment: {n_hits}/{n_total} ({pct:.0f}%)")
if drivers_found:
    print(f"  Known Drivers Found: {', '.join(drivers_found)}")
else:
    print(f"  Known Drivers Found: (novel/research targets)")

# Restore logging
_logging.root.setLevel(_prev_root_level)
_logging.getLogger("fusion_oncology").setLevel(_logging.WARNING)
_logging.getLogger("transformers").setLevel(_logging.WARNING)

# FINAL OUTPUT
print("\n" + "=" * 70)
print("INFERENCE COMPLETE - All 9 pipeline stages executed successfully")
print("=" * 70)
print(f"\nSUMMARY:")
print(f"  Sample: GDSC cell line ({true_tissue})")
print(f"  Top Target: {top_genes[0]}")
print(f"  Fusion Index: {fusion_results.iloc[0]['Fusion_Index']:.4f}")
print(f"  Evidence Score: {gene_evidence.get('evidence_score', 0):.3f}/1.0")
print(
    f"  Available Drugs: {top_drugs if top_drugs != '—' else 'Research target (no approved drugs)'}"
)
print(f"  Predicted Response: {sim_summary['recist']}")
print(f"  Output saved to: /kaggle/working/results/")
print("\nReady for clinical validation and wet lab experiments")

---

## What This Notebook Delivers

### Complete Capabilities Demonstrated:

1. **Production-Grade Multi-Modal Fusion Model**
   - XGBoost baseline classifier trained on GDSC drug sensitivity (LN_IC50) to identify top-K gene importances
   - 10 row-level distributional features per sample (mean, std, skew, kurtosis, IQR, CV, etc.)
   - DNABERT-2 sequence embeddings (768-dim) computed for top-K genes
   - Sensitivity-weighted genomic context: each gene's embedding weighted by per-cell-line drug sensitivity
   - Single XGBoost trained on concatenated (drug sensitivity + 10 engineered + 768 DNABERT-2) feature space
   - ONE unified set of CV metrics (Accuracy, Precision, Recall, F1, F2, ROC AUC)
   - Repeated stratified 5-fold CV (3 repeats for ≥200 samples) for stable metric estimates
   - Optional Optuna Bayesian HPO (50-trial search maximising weighted F1)
   - Fusion Index = Importance × Instability × 1000 for target ranking
   - Automated gene-symbol filtering (regex-based) to exclude non-gene targets
   - Intelligent class merging: cancer types with < 40 cell lines → "OTHER" for robust CV
   - Cancer driver enrichment against COSMIC Cancer Gene Census (62 reference genes)

2. **DNABERT-2 Integration**
   - Revision-pinned model loading (commit SHA lock) to prevent untrusted code injection
   - Flash attention compatibility patch for Triton ≥ 3.0
   - Batch and single-sequence embedding with `outputs[0]` tuple handling
   - Attention-masked mean pooling (padding tokens excluded from embeddings)
   - 768-dim embeddings fused into XGBoost feature space (not used in isolation)

3. **Clinical Evidence Integration**
   - OpenTargets disease association scores
   - ClinicalTrials.gov active study counts
   - CIViC clinical interpretations
   - Composite evidence scoring

4. **Drug-Target Mapping**
   - FDA-approved therapy lookup per gene target
   - Druggability assessment
   - Empty-state guard when no drugs match top targets

5. **Resistance Prediction**
   - Known resistance mutations per target gene
   - Compensatory pathway activation detection
   - Risk level stratification (High / Medium / Low)

6. **Digital Twin Simulations**
   - Gompertzian tumor growth ODE model
   - Physically calibrated immune kill rate (1e-9 day⁻¹)
   - Sensitive vs resistant cell population tracking
   - RECIST response classification (CR / PR / SD / PD)
   - Drug regimen scheduling with resistance emergence

7. **Companion Diagnostics**
   - Patient-specific mutation profiling
   - AMP/ASCO/CAP tier classifications
   - Treatment recommendations with confidence scores

8. **Pathway Enrichment**
   - KEGG cancer pathway mapping
   - Empty-state fallback when no pathways match

9. **Visualization Dashboard**
   - Fusion Index bar chart ranking
   - Pathway and drug availability charts with empty-state guards
   - Importance vs Instability scatter with normalized bubble sizes and gene annotations
   - Tumor volume and cell population time-series plots

### Data Pipeline:

- **Input**: GDSC drug sensitivity data (1,002 cell lines × 265 drugs)
- **Pivot**: TARGET column → gene-level feature matrix (LN_IC50 per gene)
- **Filter**: Regex `^[A-Z][A-Z0-9/.:-]*` keeps gene symbols, drops descriptive terms
- **Merge**: Cancer types with < 40 cell lines merged into OTHER for CV stability
- **Engineer**: 10 row-level distributional features (mean, std, skew, kurtosis, IQR, CV) per sample
- **Fusion**: DNABERT-2 embeddings weighted by drug sensitivity, concatenated with original + engineered features
- **Model**: Single XGBoost classifier on combined (N + 10 + 768) feature space
- **CV**: Repeated stratified 5-fold × 3 repeats for stable metric estimates
- **Output**: Ranked therapeutic targets with multi-modal Fusion Index scores

### Output Files (saved to `/kaggle/working/results/`):

- `fusion_results.csv` - Ranked gene targets with Fusion Index scores
- `clinical_evidence.json` - Aggregated evidence per gene
- `drug_annotations.csv` - Drug-target mappings
- `resistance_report.csv` - Known resistance mechanisms
- `digital_twin_trajectory.csv` - Simulation time series
- `companion_dx_report.json` - Patient treatment recommendations

---

### Important Disclaimer

**This is a research tool for hypothesis generation, NOT a clinical diagnostic device.**

- Requires experimental validation in lab (CRISPR, RNAi, drug screens)
- Does not replace expert clinical judgment
- Not FDA approved for clinical decision-making
- Results should be validated in independent cohorts
- GDSC cell lines are lab-adapted models; results require clinical translation
- Consult with oncologists and regulatory experts before clinical use

**Intended Users:** Cancer researchers, bioinformaticians, computational biologists, precision oncology teams

---

## Resources and Citation

**GitHub Repository:**  
[github.com/mytechnotalent/fusion_oncology](https://github.com/mytechnotalent/fusion_oncology)

**Documentation:**  
See README for full CLI reference and API documentation

**Citation:**
```bibtex
@software{fusion_oncology_2026,
  title={Fusion Oncology: Multi-Modal AI for Cancer Target Discovery},
  author={Thomas, Kevin},
  year={2026},
  url={https://github.com/mytechnotalent/fusion_oncology},
  note={Combines XGBoost, DNABERT-2, and clinical evidence for therapeutic target identification}
}
```

**License:** MIT | **Contact:** [@mytechnotalent](https://github.com/mytechnotalent)

---

**Questions or issues?** Open an issue on GitHub or contribute via pull request!